# Security Extension for Data Minimization

This notebook demonstrates the security features added to the minimization module, closing gaps
identified by Goldsteen et al. (2021):

1. **Sensitivity-weighted NCP:** up-weights sensitive features in the information-loss metric.
2. **PA-ILAG scoring:** integrates attribute-disclosure AUC into feature-removal decisions.
3. **Cell post-processing:** enforces l-diversity and replaces representatives with DP-noised synthetic points.

Set `DATASET` in the cell below to choose the dataset:

In [1]:
DATASET = 'german'   # 'synthetic', 'adult', or 'german'

## Imports, dataset choice, and parallelism patch

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, time
import numpy as np
import pandas as pd

_nb_dir = os.path.dirname(os.path.abspath('__file__'))
_repo_root = os.path.abspath(os.path.join(_nb_dir, '..'))
os.chdir(_repo_root)
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.metrics import roc_auc_score as _roc
from sklearn.model_selection import train_test_split as _tts

from apt.minimization import GeneralizeToRepresentative
from apt.utils.datasets import ArrayDataset
from apt.utils.dataset_utils import get_adult_dataset_pd, get_german_credit_dataset_pd
import apt.minimization.security_metrics as _sec_mod

# Parallelism patch: only applied for real datasets where PA-ILAG is the bottleneck.
if DATASET != 'synthetic':
    def _parallel_sensitive_auc(data, sensitive_feature, test_size=0.3, random_state=42):
        """Parallelised AUC attacker: LogisticRegression(n_jobs=-1)."""
        target = data[sensitive_feature]
        if not pd.api.types.is_numeric_dtype(target):
            target, _ = pd.factorize(target, sort=True)
        else:
            target = target.to_numpy()
        predictors = pd.get_dummies(data.drop(columns=[sensitive_feature]), drop_first=False)
        x_tr, x_te, y_tr, y_te = _tts(
            predictors, target, test_size=test_size, random_state=random_state, stratify=target)
        clf = _LR(max_iter=800, solver='lbfgs', n_jobs=-1)
        clf.fit(x_tr, y_tr)
        proba = clf.predict_proba(x_te)
        if proba.shape[1] == 2:
            return float(_roc(y_te, proba[:, 1]))
        return float(_roc(y_te, proba, multi_class='ovr'))

    _sec_mod.compute_sensitive_auc = _parallel_sensitive_auc
    print('Parallelism: compute_sensitive_auc patched with n_jobs=-1.')

print('Selected dataset:', DATASET)

Parallelism: compute_sensitive_auc patched with n_jobs=-1.
Selected dataset: german


## Load dataset and build encoder

In [3]:
if DATASET == 'synthetic':
    rng = np.random.RandomState(42)
    n = 400
    age   = rng.randint(18, 70, size=n)
    hours = rng.randint(10, 60, size=n)
    sensitive = (age > 40).astype(int)
    score  = age + hours // 2 + sensitive * 5
    labels = (score > np.median(score)).astype(int)
    x_train = pd.DataFrame({'age': age, 'hours': hours, 'sensitive': sensitive})
    x_test  = x_train
    y_train = labels;  y_test = labels
    sensitive_feature = 'sensitive'
    target_accuracy   = 0.99
    auc_threshold     = 0.8
    cat_cols          = []
    encoder           = None
    dt_params         = {'random_state': 0, 'min_samples_split': 2, 'min_samples_leaf': 1}

elif DATASET == 'adult':
    (x_train, y_train), (x_test, y_test) = get_adult_dataset_pd()
    sensitive_feature = 'sex'
    target_accuracy   = 0.98
    auc_threshold     = 0.7
    dt_params         = {'random_state': 0}

elif DATASET == 'german':
    (x_train, y_train), (x_test, y_test) = get_german_credit_dataset_pd()
    sensitive_feature = 'Personal_status_sex'
    target_accuracy   = 0.97
    auc_threshold     = 0.7
    dt_params         = {'random_state': 0}

else:
    raise ValueError('DATASET must be "synthetic", "adult", or "german"')

feat_names = x_train.columns.tolist()

if DATASET != 'synthetic':
    num_cols = x_train.select_dtypes(include='number').columns.tolist()
    cat_cols = [c for c in feat_names if c not in num_cols]
    encoder  = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), cat_cols),
        ]
    )
    encoder.fit(x_train)
    print('Numeric features :', num_cols)
    print('Categorical feats:', cat_cols)

print('Dataset          :', DATASET)
print('Train shape      :', x_train.shape, '  Test:', x_test.shape)
print('Sensitive feature:', sensitive_feature)
print('Target accuracy  :', target_accuracy)
x_train.head()

Numeric features : ['Duration_in_month', 'Credit_amount', 'Installment_rate', 'Present_residence', 'Age', 'Number_of_existing_credits', 'N_people_being_liable_provide_maintenance', 'Telephone', 'Foreign_worker']
Categorical feats: ['Existing_checking_account', 'Credit_history', 'Purpose', 'Savings_account', 'Present_employment_since', 'Personal_status_sex', 'debtors', 'Property', 'Other_installment_plans', 'Housing', 'Job']
Dataset          : german
Train shape      : (700, 20)   Test: (300, 20)
Sensitive feature: Personal_status_sex
Target accuracy  : 0.97


,Existing_checking_account,Duration_in_month,Credit_history,Purpose,Credit_amount,Savings_account,Present_employment_since,Installment_rate,Personal_status_sex,debtors,Present_residence,Property,Age,Other_installment_plans,Housing,Number_of_existing_credits,Job,N_people_being_liable_provide_maintenance,Telephone,Foreign_worker
0,A14,24,A32,A41,7814,A61,A74,3,A93,A101,3,A123,38,A143,A152,1,A174,1,1,1
1,A14,33,A33,A49,2764,A61,A73,2,A92,A101,2,A123,26,A143,A152,2,A173,1,1,1
2,A11,9,A32,A42,2136,A61,A73,3,A93,A101,2,A121,25,A143,A152,1,A173,1,0,1
3,A14,28,A34,A43,2743,A61,A75,4,A93,A101,2,A123,29,A143,A152,2,A173,1,0,1
4,A11,24,A33,A43,1659,A61,A72,4,A92,A101,2,A123,29,A143,A151,1,A172,1,1,1


## Train target model

For real datasets the model is trained on the **encoded** feature matrix (one-hot categorical columns).
`GeneralizeToRepresentative` receives the **raw** DataFrame and the `encoder` object.
For the synthetic dataset no encoding is needed.

In [4]:
x_train_enc = encoder.transform(x_train) if encoder is not None else x_train.values
x_test_enc  = encoder.transform(x_test)  if encoder is not None else x_test.values

est = DecisionTreeClassifier(**dt_params)
est.fit(x_train_enc, y_train)

train_preds = est.predict(x_train_enc)
test_preds  = est.predict(x_test_enc)

print('Target model train accuracy: %.4f' % est.score(x_train_enc, y_train))
print('Target model test  accuracy: %.4f' % est.score(x_test_enc,  y_test))


def score_out(out):
    """Score generalized output against test_preds, encoding if needed."""
    enc = encoder.transform(out) if encoder is not None else np.asarray(out)
    return float(est.score(enc, test_preds))

Target model train accuracy: 1.0000
Target model test  accuracy: 0.6933


## Baseline minimization (no security features)

Establishes the accuracy floor and the generalizations the paper's method alone would produce.

In [5]:
t0 = time.time()

baseline = GeneralizeToRepresentative(
    estimator=est,
    target_accuracy=target_accuracy,
    categorical_features=cat_cols or None,
    encoder=encoder,
    features_to_minimize=feat_names,
)
baseline.fit(dataset=ArrayDataset(x_train, train_preds, features_names=feat_names))
base_out = baseline.transform(None, dataset=ArrayDataset(x_test, features_names=feat_names))
base_acc = score_out(base_out)

print('Baseline fit: %.1f s' % (time.time() - t0))
print('Baseline accuracy on generalized data: %.4f' % base_acc)
print('Baseline generalizations:', baseline.generalizations)

Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.682143
Improving accuracy
feature to remove: Personal_status_sex
Removed feature: Personal_status_sex, new relative accuracy: 0.675000
feature to remove: Job
Removed feature: Job, new relative accuracy: 0.635714
feature to remove: Housing
Removed feature: Housing, new relative accuracy: 0.635714
feature to remove: Existing_checking_account
Removed feature: Existing_checking_account, new relative accuracy: 0.667857
feature to remove: Purpose
Removed feature: Purpose, new relative accuracy: 0.710714
feature to remove: Age
Removed feature: Age, new relative accuracy: 0.710714
feature to remove: Present_residence
Removed feature: Present_residence, new relative accuracy: 0.703571
feature to remove: Installment_rate
Removed feature: Installment_rate, new relative accuracy: 0.696429
feature to remove: N_people_being_liable_provide_maintenance
R

## Security-extended minimization

All five security parameters are active:

| Parameter | Effect |
|---|---|
| `weighted_ncp_config` | Up-weights sensitive feature in NCP (α = 2.0) |
| `pa_ilag_config` | PA-ILAG: defers un-generalizing leaky features |
| `disclosure_config` | Measures AUC before/after generalization |
| `diversity_config` | Enforces k ≥ 5 records and l ≥ 2 sensitive values per cell |
| `dp_config` | Replaces representatives with DP-noised synthetic points (ε = 1.0) |

> **Runtime note (real datasets):** PA-ILAG calls the logistic-regression attacker for every
> candidate feature at every removal step. On Adult this is roughly 12 features × ~10 steps = ~120
> LR fits on 32 k rows. With `n_jobs=-1` expect **5–15 min**. German Credit: < 2 min.

In [6]:
t0 = time.time()
print('Fitting security-extended minimizer - watch security_verbose output below...')

secure = GeneralizeToRepresentative(
    estimator=est,
    target_accuracy=target_accuracy,
    categorical_features=cat_cols or None,
    encoder=encoder,
    features_to_minimize=feat_names,
    # Feature 1: sensitive features carry higher NCP weight so they are
    # kept generalised for longer before being un-generalised.
    weighted_ncp_config={'sensitive_features': [sensitive_feature], 'alpha': 2.0},
    # Feature 2a: PA-ILAG penalises features whose un-generalisation
    # increases the attacker's AUC on the sensitive attribute.
    pa_ilag_config={'lambda_attr': 1.0, 'sensitive_feature': sensitive_feature},
    # Feature 2b: measure attribute-disclosure AUC before and after.
    disclosure_config={
        'sensitive_feature': sensitive_feature,
        'auc_threshold': auc_threshold,
        'test_size': 0.3,
        'seed': 42,
    },
    # Feature 3: enforce cell privacy stack (k/l-diversity + t-closeness).
    diversity_config={
        'sensitive_features': [sensitive_feature],
        'k_min': 5,
        'l_min': 2,
    },
    # Feature 4: replace real training records used as representatives
    # with DP-noised synthetic points (Laplace/exponential mechanism).
    dp_config={'epsilon': 1.0, 'max_retries': 10, 'seed': 42},
    security_verbose=True,
)
secure.fit(dataset=ArrayDataset(x_train, train_preds, features_names=feat_names))
sec_out = secure.transform(None, dataset=ArrayDataset(x_test, features_names=feat_names))
sec_acc = score_out(sec_out)

elapsed = time.time() - t0
print('\nSecurity fit: %.1f s (%.1f min)' % (elapsed, elapsed / 60))
print('Security accuracy on generalized data: %.4f' % sec_acc)

Fitting security-extended minimizer - watch security_verbose output below...
Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.682143
Improving accuracy
feature to remove: Job
Removed feature: Job, new relative accuracy: 0.642857
feature to remove: Housing
Removed feature: Housing, new relative accuracy: 0.642857
feature to remove: Personal_status_sex
Removed feature: Personal_status_sex, new relative accuracy: 0.635714
feature to remove: Existing_checking_account
Removed feature: Existing_checking_account, new relative accuracy: 0.667857
feature to remove: Purpose
Removed feature: Purpose, new relative accuracy: 0.710714
feature to remove: Age
Removed feature: Age, new relative accuracy: 0.710714
feature to remove: Present_residence
Removed feature: Present_residence, new relative accuracy: 0.703571
feature to remove: Installment_rate
Removed feature: Installment_rate, new relative accu

## Security report

In [7]:
rpt  = secure.security_report or {}
disc = rpt.get('disclosure') or {}
div  = rpt.get('diversity') or {}
dp   = rpt.get('randomized_representatives') or {}
pa   = rpt.get('pa_ilag') or {}

print('=' * 64)
print('  Security Extension Results  --  %s' % DATASET)
print('=' * 64)

print('\n1) Accuracy preservation')
print('   target : %.4f' % target_accuracy)
print('   baseline (no security): %.4f' % base_acc)
print('   secure  (all features): %.4f' % sec_acc)
print('   accuracy cost of security: %.4f' % (base_acc - sec_acc))

print('\n2) Attribute disclosure  (lower AUC = less leakage = better)')
print('   sensitive feature : %s' % sensitive_feature)
print('   AUC before        : %s' % disc.get('auc_before', 'n/a'))
print('   AUC after         : %s' % disc.get('auc_after', 'n/a'))
print('   AUC delta         : %s' % disc.get('auc_delta', 'n/a'))
print('   threshold (<= %.1f): %s' % (auc_threshold, 'PASS' if disc.get('threshold_pass') else 'FAIL'))

print('\n3) Diversity / homogeneity-attack mitigation')
print('   cells before / after merge : %s / %s' % (
       div.get('num_cells_before', 'n/a'), div.get('num_cells_after', 'n/a')))
print('   k-violations before / after: %s / %s' % (
       div.get('k_violations_before', 'n/a'), div.get('k_violations_after', 'n/a')))
print('   l-violations before / after: %s / %s' % (
       div.get('l_violations_before', 'n/a'), div.get('l_violations_after', 'n/a')))
print('   merges performed           : %s' % div.get('num_merges', 'n/a'))

print('\n4) DP representative randomization')
print('   non-verbatim rate: %.2f' % dp.get('non_verbatim_rate', 0.0))
print('   fallback count   : %s'   % dp.get('fallback_count', 'n/a'))

print('\n5) PA-ILAG vs plain ILAG feature removal order')
print('   PA-ILAG order    :', pa.get('feature_removal_order', []))
print('   plain ILAG order :', pa.get('baseline_removal_order', []))
print('   divergent steps  :', pa.get('num_divergence_steps', 0))
print('=' * 64)

  Security Extension Results  --  german

1) Accuracy preservation
   target : 0.9700
   baseline (no security): 0.7333
   secure  (all features): 0.7233
   accuracy cost of security: 0.0100

2) Attribute disclosure  (lower AUC = less leakage = better)
   sensitive feature : Personal_status_sex
   AUC before        : 0.6433576827088481
   AUC after         : 0.653281560270059
   AUC delta         : 0.009923877561210936
   threshold (<= 0.7): PASS

3) Diversity / homogeneity-attack mitigation
   cells before / after merge : 88 / 70
   k-violations before / after: 84 / 69
   l-violations before / after: 84 / 69
   merges performed           : 18

4) DP representative randomization
   non-verbatim rate: 1.00
   fallback count   : 0

5) PA-ILAG vs plain ILAG feature removal order
   PA-ILAG order    : ['Job', 'Housing', 'Personal_status_sex', 'Existing_checking_account', 'Purpose', 'Age', 'Present_residence', 'Installment_rate', 'N_people_being_liable_provide_maintenance', 'Foreign_worker'